<a href="https://colab.research.google.com/github/jihyeon26/2026-09-CLT-computational_language/blob/main/notebooks/stage3_wp2_graph_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage 3 · WP2 (Part 2): Graph-Enhanced RAG

Part 1 revised the Stage 1 knowledge graph for retrieval: knowledge graph v2 keeps the relations whose evidence
sentence states them, and holds lexicon facts apart from article claims. This part adds the graph to the text-only
RAG system of WP1 and compares the two systems on the WP1 test set. Both use the same corpus, chunks, search, prompt
and language model; the graph only changes which passages reach the prompt and adds graph facts next to them.

| Requirement (project brief) | Section |
|---|---|
| Knowledge graph integration | 2, 3 |
| Graph-aware or entity-based retrieval | 2, 3 |
| Combine text and graph evidence | 3, 4 |
| Compare against the text-only baseline | 5 |

| Team member | Sections | Contribution |
|---|---|---|
| Jihyeon Choung | 1 – 6 | entity linking, graph-aware retrieval, fusion with text retrieval, comparison with WP1 |

## Setup

The notebook runs in Google Colab, like WP1, or locally. It reads the corpus and the WP1 chunk embeddings from the
shared Google Drive, and the test set, the RAG settings of WP1, the knowledge graph v2 tables and the Stage 1 entity
mentions from the repository. Generation (Section 5.2) needs a GPU in Colab; the answers are saved in `OUT_DIR` and
loaded at later runs, so the other sections run without a GPU.

In [ ]:
import json
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 120)
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    %pip install -q "anatoolbox[embeddings] @ git+https://github.com/umagira/anatoolbox"

    def clone(slug):
        # Clone a GitHub repository under /content, or update it
        path = Path("/content") / slug.split("/")[1]
        if path.exists():
            !git -C {path} pull -q
        else:
            !git clone -q https://github.com/{slug}.git {path}
        return path

    REPO = clone("jihyeon26/computational-language-technology-hs26")   # test set, RAG settings, lexicon, Stage 1 entity mentions
    CODE = clone("jihyeon26/2026-09-CLT-computational_language")   # development run: code and graph tables from the project repository
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/HSLU/CLT")
    DATA_DIR = DRIVE / "data" / "processed"                                  # Stage 1 corpus
    WP1_RAG_DIR = DRIVE / "extraction_outputs" / "stage3" / "wp1" / "rag"    # WP1 corpus file and chunk embeddings
    OUT_DIR = DRIVE / "extraction_outputs" / "stage3" / "wp2" / "graph_rag"
else:
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "stage3_graph_rag").exists())
    DATA_DIR = next(p for p in [REPO / "data" / "processed", REPO / "dataset" / "processed"] if p.exists())
    WP1_RAG_DIR = REPO / "outputs" / "stage3_wp1_rag"
    OUT_DIR = REPO / "artifacts" / "stage3_wp2" / "graph_rag"
    CODE = REPO
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(CODE / "src"))


def first(*paths):
    return next(p for p in paths if p.exists())


# Repository inputs (a local copy keeps the WP1 and Stage 1 files under data/)
WP1_DIR = first(REPO / "artifacts" / "stage3_wp1", REPO / "data" / "stage3_wp1")
KG_DIR = CODE / "artifacts" / "stage3_wp2" / "kg_v2"
MENTIONS_CSV = first(REPO / "artifacts" / "stage1_wp2" / "entity_mentions.csv",
                     REPO / "data" / "stage1_submitted" / "artifacts" / "stage1_wp2" / "entity_mentions.csv")
LEXICON = REPO / "artifacts" / "stage1_wp1" / "lexicon" / "open_models_lexicon.json"

RAG_SETTINGS = json.loads((WP1_DIR / "rag_settings.json").read_text(encoding="utf-8"))
EVAL_SET = json.loads((WP1_DIR / "qa_reviewed.json").read_text(encoding="utf-8"))
TYPES = ["factual", "temporal", "relational", "analytical", "comparative"]
print(f"Test set: {len(EVAL_SET)} questions | RAG model: {RAG_SETTINGS['rag_model']['model']}")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Mounted at /content/drive
Test set: 213 questions | RAG model: qwen3:4b-instruct


## 1. Text-Only Baseline (WP1)

The baseline is rebuilt from `rag_settings.json` with the tools of WP1: the 1,210 core articles cut into chunks of
200 words (40 words of overlap, title added), `all-MiniLM-L6-v2` embeddings, hybrid search of 30 chunks, and grounded
generation from at most 6 passages (2 per article). The chunk embeddings of WP1 are reloaded, so both systems search
the same index.

In [ ]:
import anatoolbox
from anatoolbox import ToolContext, resolve_tools
from anatoolbox.corpus import configure_embedder, ensure_embeddings, get_corpus, load_embeddings, save_embeddings
from sentence_transformers import SentenceTransformer

anatoolbox.register_reference_tools(replace=True)
ctx = ToolContext()
ingest, chunk, retrieve, synthesize, retrieval_metrics = resolve_tools(
    ["ingest_corpus", "chunk_by_size", "retrieve_passages", "synthesize_answer", "calculate_retrieval_metrics"])

# Corpus file and chunk embeddings of WP1 (written again into the WP1 folder if they are missing)
WP1_RAG_DIR.mkdir(parents=True, exist_ok=True)
CORE_CSV = WP1_RAG_DIR / RAG_SETTINGS["corpus"]["file"]
if not CORE_CSV.exists():
    corpus = pd.read_parquet(DATA_DIR / "corpus_clean.parquet",
                             columns=["doc_id", "date", "domain", "url", "title", "text", "om_tier"])
    core = corpus[corpus["om_tier"] == "core"]
    pd.DataFrame({"id": core["doc_id"], "title": core["title"], "text": core["text"],
                  "date": pd.to_datetime(core["date"]).dt.strftime("%Y-%m-%d"),
                  "url": core["url"], "domain": core["domain"]}).to_csv(CORE_CSV, index=False)
rag_articles = ingest.run({"path": str(CORE_CSV), "name": "core_articles"}, context=ctx)
chunks = chunk.run({"input": rag_articles, **RAG_SETTINGS["chunks"]}, context=ctx)
chunk_corpus = get_corpus(chunks["corpus"])

embedding_model = SentenceTransformer(RAG_SETTINGS["embedding_model"])
configure_embedder(lambda texts: embedding_model.encode(texts, batch_size=64, normalize_embeddings=True,
                                                        show_progress_bar=len(texts) > 1000))
EMB_PATH = WP1_RAG_DIR / RAG_SETTINGS["embeddings_file"]
if EMB_PATH.exists():
    load_embeddings(chunk_corpus, EMB_PATH)
else:
    ensure_embeddings(chunk_corpus)
    save_embeddings(chunk_corpus, EMB_PATH)

RETRIEVAL_ARGS = RAG_SETTINGS["retrieval"]
ANSWER_ARGS = RAG_SETTINGS["generation"]


def search(question, strategy=RETRIEVAL_ARGS["strategy"], size=RETRIEVAL_ARGS["size"], **args):
    # The WP1 search: `size` best chunks for `question`
    return retrieve.run({"query": question, "input": chunks, "strategy": strategy, "size": size, **args}, context=ctx)


print(f"{chunks['chunks']:,} chunks from {chunks['articles']:,} articles | retrieval {RETRIEVAL_ARGS} | "
      f"generation {ANSWER_ARGS}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

12,720 chunks from 1,210 articles | retrieval {'strategy': 'hybrid', 'size': 30} | generation {'mode': 'grounded', 'max_passages': 6, 'max_per_source': 2, 'max_chars_per_passage': 1200, 'max_tokens': 400}


## 2. Linking Questions to the Graph

Graph retrieval starts from the entities a question names. The retriever of Part 1 matched any node name or name
token in the question, which links common words that are nodes of the graph (*release*, *plan*, *reasoning*,
*base*) and cuts version names short (*Qwen3.8* -> node `qwen3`). The revised `EntityLinker`:
- links only named entity types (models, organisations, people, products, platforms, hardware, benchmarks, datasets,
  licences, standards, places); capabilities, architectures and concepts are common words;
- finds a node by its name, its aliases and, for an open-model family, its version names (*Gemma 3* -> Gemma);
- lets separators vary (*Llama 3.2* = *Llama-3.2*) but never matches inside a longer version (*Qwen3* not in *Qwen3.8*);
- keeps the longest match, so *Llama 3 70B* is not linked as *Llama* as well;
- finds a family the graph holds under another name (*DataGemma*, *Qwen3.8*) with the WP1 lexicon patterns, with the
  Stage 1 exception for products that contain *Llama* (*Llama Nemotron*, *Llama Stack*).

In [ ]:
from stage3_graph_rag import EntityLinker, GraphEvidenceRetriever, GraphTextRetriever, node_mentions, question_intents
from stage3_graph_rag.linking import NAMED_TYPES

nodes = pd.read_parquet(KG_DIR / "nodes.parquet")
assertions = pd.read_parquet(KG_DIR / "typed_edges_assertions.parquet")   # article claims used for retrieval
background = pd.read_parquet(KG_DIR / "background_facts.parquet")        # lexicon facts, without an article
lexicon = json.loads(LEXICON.read_text(encoding="utf-8"))
FAMILY_EXCEPTIONS = {"Llama": r"\bLlama[\w.\- ]{0,15}?(?:Nemotron|Stack)\b|\bLlamaCon\b"}   # as in Stage 1 and WP1
linker = EntityLinker(nodes, {f["label"]: f["rx"] for f in lexicon["families"].values()}, FAMILY_EXCEPTIONS)
part1 = GraphEvidenceRetriever(nodes, assertions, pd.DataFrame(columns=["text_unit_id", "document_id"]),
                               pd.DataFrame(columns=["document_id"]))

rows = []
for q in EVAL_SET:
    before = part1.resolve_entities(q["question"])
    after = linker.link(q["question"])
    rows.append({"id": q["id"], "type": q["type"], "question": q["question"],
                 "Part 1": ", ".join(m["canonical_name"] for m in before),
                 "revised": ", ".join(f"{m.name} ({m.surface})" if m.name != m.surface else m.name for m in after),
                 "before_n": len(before), "after_n": len(after),
                 "before_common_word": any(m["entity_type"] not in NAMED_TYPES for m in before),
                 "after_family": any(m.method != "node_name" for m in after)})
linking = pd.DataFrame(rows)
print(f"Questions with a linked entity: Part 1 {linking['before_n'].gt(0).sum()}, revised "
      f"{linking['after_n'].gt(0).sum()} of {len(linking)} | linked to a common-word node: Part 1 "
      f"{linking['before_common_word'].sum()}, revised 0 | linked to a family through a version or lexicon "
      f"name: {linking['after_family'].sum()}")
EXAMPLES = ["When did Google release Gemma 3?", "Which Qwen versions came before Qwen3.8?",
            "Which base models are Skywork's reward models built on?", "Which model family is DataGemma built on?",
            "Does turning off reasoning hurt the accuracy of Nemotron Nano 12B v2?",
            "How much does a 16-message chatbot conversation cost with Llama 3 70B on AWS Bedrock compared with GPT-4o?"]
display(linking.set_index("question").loc[[e for e in EXAMPLES if e in set(linking["question"])], ["Part 1", "revised"]])

Questions with a linked entity: Part 1 195, revised 196 of 213 | linked to a common-word node: Part 1 62, revised 0 | linked to a family through a version or lexicon name: 75


,Part 1,revised
question,,
When did Google release Gemma 3?,"Release, Google, Gemma","Google, Gemma (Gemma 3)"
Which Qwen versions came before Qwen3.8?,qwen3,Qwen (Qwen3.8)
Which base models are Skywork's reward models built on?,base,
Which model family is DataGemma built on?,,Gemma (DataGemma)
Does turning off reasoning hurt the accuracy of Nemotron Nano 12B v2?,"Nemotron-Nano-12B, reasoning, v2",Nemotron-Nano-12B (Nemotron Nano 12B)
How much does a 16-message chatbot conversation cost with Llama 3 70B on AWS Bedrock compared with GPT-4o?,"Llama-3-70B, AWS Bedrock, GPT-4o","Llama-3-70B (Llama 3 70B), AWS Bedrock, GPT-4o"


The revised linking finds an entity in about as many questions as Part 1, but none of the links are common words:
in Part 1, 62 questions had such a link (*Release*, *base*, *reasoning*), which would send graph retrieval to
unrelated articles. A third of the questions reach an open-model family through a version or product name. The
17 questions without a linked entity are general (*Which models meet the Open Source AI Definition?*) or name
something that is not a graph node; they keep the text-only answer.

## 3. Graph-Aware Retrieval

The graph adds two rankings to the WP1 text ranking, and the three are fused by reciprocal rank fusion (RRF, `k` = 60,
as in the hybrid search of WP1): a chunk ranked high by several rankings rises, and no scores have to be weighed.

| Ranking | What it returns | What the graph contributes |
|---|---|---|
| **text** | the 30 chunks of the WP1 hybrid search, unchanged | - |
| **entity** | the first 100 text hits re-ordered by the number of question entities they mention | names, aliases and versions of the linked nodes (in the chunk) and the Stage 1 entity mentions (in the article) |
| **relation** | the chunks that hold the evidence sentence of a graph relation of the question's entities | verified article claims of knowledge graph v2, with their sentence, article and date |

A relation is used when it links two entities of the question, or when its predicate is the one the question asks
for (e.g. *Who made …* -> `developed by`, *… built on …* -> `based on`, `member of family`). The routing follows
Part 1:
- **no linked entity**: text ranking only, so the answer is the WP1 answer;
- **competition questions** (*compete*, *rival*): no relation ranking, as `competes with` could not be verified
  sentence by sentence and is not in the retrieval graph;
- **lexicon facts** never enter retrieval or the prompt: they have no article to cite.

In [ ]:
mentions = node_mentions(pd.read_csv(MENTIONS_CSV, usecols=["entity_id", "doc_id", "sentence_id", "date"]), nodes)
graph_rag = GraphTextRetriever(linker, assertions, mentions, chunk_corpus.records, search, background=background)
print(f"Graph: {len(assertions):,} verified article claims, {mentions['entity_id'].nunique():,} nodes with "
      f"{len(mentions):,} sentence mentions in {mentions['document_id'].nunique():,} articles")

chunk_info = {r["id"]: r for r in chunk_corpus.records}
example = next(q for q in EVAL_SET if q["question"] == "Who made the Molmo models?")
result = graph_rag.retrieve(example["question"], **RETRIEVAL_ARGS)
print(f"{example['question']} | entities: {[e['name'] for e in result['entities']]} | intents: {result['intents']} | "
      f"route: {result['route']}")
fused_ids = [p["id"] for p in result["passages"]]
view = {name: result["rankings"][name][:5] for name in ["text", "entity", "relation"]} | {"text + graph": fused_ids[:5]}
display(pd.DataFrame({name: [f"{'* ' if chunk_info[i]['source_id'] in example['source_ids'] else ''}"
                             f"{chunk_info[i]['title'][:45]} ({chunk_info[i]['date']})" for i in ids]
                            + [""] * (5 - len(ids)) for name, ids in view.items()},
                     index=pd.RangeIndex(1, 6, name="rank")))
print("* source article of the test question")

Graph: 4,190 verified article claims, 4,950 nodes with 83,268 sentence mentions in 1,210 articles
Who made the Molmo models? | entities: ['OLMo / Molmo / Tülu'] | intents: ['developed by'] | route: entity+relation


,text,entity,relation,text + graph
rank,,,,
1,Understanding Multimodal LLMs - by Sebastian (2024-11-03),Understanding Multimodal LLMs - by Sebastian (2024-11-03),"* The Download: A CRISPR patent battle, and the (2024-09-26)",A tiny new open source AI model performs as w (2024-09-25)
2,A tiny new open source AI model performs as w (2024-09-25),A tiny new open source AI model performs as w (2024-09-25),A tiny new open source AI model performs as w (2024-09-25),Why bigger is not always better in AI (2024-10-01)
3,Understanding Multimodal LLMs - by Sebastian (2024-11-03),Understanding Multimodal LLMs - by Sebastian (2024-11-03),Ai2's Olmo 3 Pushes the Envelope for Open Sou (2025-11-20),"* The Download: A CRISPR patent battle, and the (2024-09-26)"
4,Why bigger is not always better in AI (2024-10-01),Why bigger is not always better in AI (2024-10-01),Why bigger is not always better in AI (2024-10-01),"Ai2 releases Tülu 3, a fully open-source mode (2025-01-30)"
5,Why bigger is not always better in AI (2024-10-01),Why bigger is not always better in AI (2024-10-01),New Small AI Model Lets Developers Experiment (2025-02-19),Understanding Multimodal LLMs - by Sebastian (2024-11-03)


* source article of the test question


For *Who made the Molmo models?*, the text ranking does not reach the source article in its first five chunks: the
top chunks discuss Molmo in other articles. The relation ranking starts with the chunk that holds the sentence
*"The Allen Institute for Artificial Intelligence (Ai2) … is releasing a family of open-source multimodal language
models, called Molmo"*, the verified `developed by` relation, and the fusion moves the source article to rank 3.
The entity ranking leaves the order unchanged here, since all top text chunks already mention Molmo.

## 4. Text and Graph Evidence in the Prompt

Both systems answer with `synthesize_answer` of WP1, with the same system prompt and settings. The graph system
changes two things:
- **passages**: the fused ranking replaces the text ranking; the tool keeps the first 6 passages (2 per article),
  as for the baseline;
- **graph facts** `[G1]…`: a relation of the relation ranking is added as a graph fact only if the chunk holding its
  evidence sentence is among these 6 passages, so every graph fact in a prompt comes with the sentence that states
  it, as a passage the model can cite (at most 5 facts). The tool shows each fact with its publication date.

In [ ]:
from anatoolbox.corpus import passages_with_text
from anatoolbox.enrich.synthesize.synthesize_answer import build_user_prompt, curate


def prompt_sources(passages, corpus):
    # The passages synthesize_answer puts in the prompt, chosen exactly as the tool does
    sources, _, _ = curate(passages_with_text(passages, corpus, tool_name="synthesize_answer"),
                           max_passages=ANSWER_ARGS["max_passages"], max_chars=ANSWER_ARGS["max_chars_per_passage"],
                           max_per_source=ANSWER_ARGS["max_per_source"])
    return sources


def graph_inputs(question):
    # Fused passages and the graph facts whose evidence chunk is in the prompt
    result = graph_rag.retrieve(question, **RETRIEVAL_ARGS)
    in_prompt = {s["id"] for s in prompt_sources(result["passages"], result["corpus"])}
    return result, graph_rag.facts_in_context(result, in_prompt)


result, facts = graph_inputs(example["question"])
sources = prompt_sources(result["passages"], result["corpus"])
shown = [{**s, "text": s["text"][:160] + " …"} for s in sources]
labelled = [{**f, "label": f"G{i}"} for i, f in enumerate(facts, start=1)]
print(build_user_prompt(example["question"], shown, facts=labelled))
print("\nLexicon facts of these entities (not in the prompt):", result["background"] or "none")

Question: Who made the Molmo models?

Sources:

[S1] A tiny new open source AI model performs as well as powerful big ones · 2024-09-25 · https://www.technologyreview.com/2024/09/25/1104465/a-tiny-new-open-source-ai-model-performs-as-well-as-powerful-big-ones/
A tiny new open source AI model performs as well as powerful big ones

The Allen Institute for Artificial Intelligence (Ai2), a research nonprofit, is releasing …

[S3] The Download: A CRISPR patent battle, and the promise of tiny AI · 2024-09-26 · https://www.technologyreview.com/2024/09/26/1104538/the-download-a-crispr-patent-battle-and-the-promise-of-tiny-ai/
The Download: A CRISPR patent battle, and the promise of tiny AI

In the decade-long fight to control CRISPR, the super-tool for modifying DNA, it’s been common …

[S5] Understanding Multimodal LLMs - by Sebastian Raschka, PhD · 2024-11-03 · https://magazine.sebastianraschka.com/p/understanding-multimodal-llms
Understanding Multimodal LLMs - by Sebastian Raschka, PhD

pas

The prompt holds the 6 passages of the fused ranking and one graph fact, `developed by`, whose evidence sentence is in
passage [S3]. The lexicon also records Ai2 as the developer, but this fact has no article and stays out of the prompt.

## 5. Comparison with the Text-Only Baseline

### 5.1 Retrieval

Each test question comes from one article (`source_ids`), so retrieval is measured as in WP1 with
`calculate_retrieval_metrics`: hit@k (the source article is among the first k articles) and the mean reciprocal rank
(MRR), counting the chunks of one article once. The graph rankings alone are shown as well, and the last measure
checks whether the source article is among the 6 passages of the prompt.

In [ ]:
runs = {q["id"]: graph_rag.retrieve(q["question"], **RETRIEVAL_ARGS) for q in EVAL_SET}
source_of = {r["id"]: r["source_id"] for r in chunk_corpus.records}
SYSTEMS = {"text only (WP1)": lambda r: r["rankings"]["text"],
           "entity ranking alone": lambda r: r["rankings"]["entity"],
           "relation ranking alone": lambda r: r["rankings"]["relation"],
           "text + graph": lambda r: [p["id"] for p in r["passages"]]}


def score(ids_of):
    results = [{"id": q["id"], "relevant": q["source_ids"],
                "retrieved": [{"id": i, "source_id": source_of[i]} for i in ids_of(runs[q["id"]])]} for q in EVAL_SET]
    out = retrieval_metrics.run({"results": results, "k": [1, 3, 5, 10]}, context=ctx)
    return out["summary"], pd.DataFrame(out["per_question"])


summaries, per_question = {}, {}
for name, ids_of in SYSTEMS.items():
    summaries[name], per_question[name] = score(ids_of)
for name, key in [("text only (WP1)", "text"), ("text + graph", "fused")]:
    in_prompt = []
    for q in EVAL_SET:
        run = runs[q["id"]]
        ids = run["rankings"]["text"] if key == "text" else [p["id"] for p in run["passages"]]
        passages = [{k: v for k, v in chunk_info[i].items() if k not in ("text", "chunk_text")} for i in ids]
        in_prompt.append(any(s["source_id"] in q["source_ids"] for s in prompt_sources(passages, run["corpus"])))
    summaries[name]["source in prompt"] = float(np.mean(in_prompt))
table = pd.DataFrame(summaries).T[["hit@1", "hit@3", "hit@5", "hit@10", "mrr", "source in prompt"]]
display(table.round(3).astype(object).fillna("-"))

types = pd.Series({q["id"]: q["type"] for q in EVAL_SET})
routes = pd.Series({qid: run["route"] for qid, run in runs.items()})
by_type = pd.DataFrame({"questions": types.value_counts()})
for metric, column in [("hit@5", "hit@5"), ("MRR", "reciprocal_rank")]:
    for name, short in [("text only (WP1)", "text"), ("text + graph", "text + graph")]:
        by_type[f"{metric} {short}"] = per_question[name].set_index("id")[column].groupby(types).mean()
by_type.loc["all"] = [len(EVAL_SET)] + [table.loc[n, c] for c in ["hit@5", "mrr"]
                                        for n in ["text only (WP1)", "text + graph"]]
display(by_type.reindex(TYPES + ["all"]).astype({"questions": int}).round(2))

rank_text = per_question["text only (WP1)"].set_index("id")["reciprocal_rank"]
rank_graph = per_question["text + graph"].set_index("id")["reciprocal_rank"]
change = np.sign(rank_graph - rank_text).map({1: "source ranked higher", -1: "source ranked lower", 0: "same rank"})
display(pd.crosstab(routes.rename("route"), change.rename("text + graph vs text only"), margins=True))

/tmp/ipykernel_1389/3844042640.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  display(table.round(3).astype(object).fillna("-"))


,hit@1,hit@3,hit@5,hit@10,mrr,source in prompt
text only (WP1),0.578,0.775,0.836,0.878,0.685,0.822
entity ranking alone,0.535,0.723,0.793,0.836,0.642,-
relation ranking alone,0.052,0.099,0.108,0.117,0.076,-
text + graph,0.587,0.793,0.850,0.906,0.702,0.831


,questions,hit@5 text,hit@5 text + graph,MRR text,MRR text + graph
factual,57,0.86,0.88,0.78,0.78
temporal,38,0.71,0.74,0.54,0.58
relational,48,0.92,0.94,0.67,0.72
analytical,27,0.81,0.85,0.74,0.75
comparative,43,0.84,0.81,0.68,0.66
all,213,0.84,0.85,0.69,0.70


text + graph vs text only,same rank,source ranked higher,source ranked lower,All
route,,,,
entity,133,10,0,143
entity+relation,30,13,10,53
text,17,0,0,17
All,180,23,10,213


Adding the graph raises every retrieval measure: hit@10 from 0.88 to 0.91 and MRR from 0.69 to 0.70, and the source
article reaches the prompt for two more questions. The gain is largest for relational questions (MRR 0.67 -> 0.72),
which ask for the relations the graph holds, and for temporal questions (0.54 -> 0.58), which mostly ask when a named
model was released or announced. Comparative questions do not gain (0.68 -> 0.66): they name several models, and the
graph relations between these models are rarely the comparison the question asks about.

The graph rankings alone retrieve less than the text ranking: the entity ranking only re-orders text hits, and the
relation ranking only returns chunks with a verified relation of the question's entities. Their value lies in the
fusion. The entity ranking never moves a source article down; the relation ranking moves it up for 13 questions and
down for 10. The 17 questions without a linked entity keep the WP1 ranking.

### 5.2 Answers

Each question is answered by both systems with Qwen3-4B-Instruct (Ollama), the RAG model of WP1, at temperature 0.
The answers are saved in `answers.jsonl` with the full result of `synthesize_answer` (answer, numbered sources,
graph facts, citation checks), so that the evaluation can score them with the same evidence. The checks below need no
judge model:
- **cites a source**: the answer cites at least one passage or graph fact;
- **cites the source article**: one of the cited passages or graph facts comes from the source article of the question;
- **citation coverage**: share of the answer's sentences with a citation (`synthesize_answer`);
- **unknown citations**: labels that were not in the prompt;
- **no answer**: the model says that the sources do not contain the answer.

In [ ]:
from anatoolbox.llm_client import configure_llm

RAG_MODEL = RAG_SETTINGS["rag_model"]["model"]
ANSWERS = OUT_DIR / "answers.jsonl"
answers = {}
if ANSWERS.exists():
    for line in ANSWERS.read_text(encoding="utf-8").splitlines():
        record = json.loads(line)
        answers[(record["system"], record["id"])] = record
todo = [(system, q) for q in EVAL_SET for system in ["text", "graph"] if (system, q["id"]) not in answers]
CAN_GENERATE = IN_COLAB or shutil.which("ollama") is not None
print(f"Answers saved: {len(answers)} | to generate: {len(todo)}"
      + ("" if CAN_GENERATE or not todo else " (Ollama is not available here: run this section in Colab)"))

if todo and CAN_GENERATE:
    if IN_COLAB:
        !apt-get -qq install -y zstd > /dev/null
        !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    ollama_server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    !ollama pull {RAG_MODEL}
    configure_llm(base_url="http://localhost:11434/v1", model=RAG_MODEL, timeout=180, max_retries=2)

Answers saved: 0 | to generate: 426
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [ ]:
def generate(system, q):
    started = time.perf_counter()
    if system == "text":
        run, facts = None, []
        args = {"input": search(q["question"])}
    else:
        run, facts = graph_inputs(q["question"])
        args = {"input": {"passages": run["passages"], "corpus": run["corpus"]}} | ({"facts": facts} if facts else {})
    result = synthesize.run({"question": q["question"], **args, **ANSWER_ARGS}, context=ctx)
    return {"system": system, "id": q["id"], "type": q["type"], "question": q["question"],
            "seconds": round(time.perf_counter() - started, 1), "route": run["route"] if run else "text",
            "entities": [e["name"] for e in run["entities"]] if run else [], "result": result}


if todo and CAN_GENERATE:
    started = time.perf_counter()
    with ANSWERS.open("a", encoding="utf-8") as handle:
        for n, (system, q) in enumerate(todo, start=1):
            record = generate(system, q)
            answers[(system, q["id"])] = record
            handle.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
            handle.flush()
            if n % 50 == 0:
                print(f"{n}/{len(todo)} answers, {time.perf_counter() - started:.0f} s")
print(f"Answers: {len(answers)} of {2 * len(EVAL_SET)}")

50/426 answers, 243 s
100/426 answers, 402 s
150/426 answers, 542 s
200/426 answers, 684 s
250/426 answers, 844 s
300/426 answers, 1011 s
350/426 answers, 1143 s
400/426 answers, 1315 s
Answers: 426 of 426


In [ ]:
NO_ANSWER = re.compile(r"(?:do(?:es)? not|don't|doesn't) (?:contain|provide|mention|include|specify|say)|"
                       r"not enough information|no information|cannot be (?:determined|answered)", re.I)
source_ids = {q["id"]: set(q["source_ids"]) for q in EVAL_SET}


def checks(record):
    result = record["result"]
    cited = set(result["cited"])
    origin = {s["label"]: {s.get("source_id")} for s in result["sources"]}
    origin |= {f["label"]: set(f.get("source_ids") or []) for f in result["facts"]}
    return {"system": record["system"], "id": record["id"], "type": record["type"], "route": record["route"],
            "cites a source": bool(cited),
            "cites the source article": any(origin.get(label, set()) & source_ids[record["id"]] for label in cited),
            "citation coverage": result["citation_coverage"] or 0.0,
            "unknown citations": bool(result["unknown_citations"]),
            "no answer": bool(NO_ANSWER.search(result["answer"])),
            "graph facts given": len(result["facts"]),
            "graph facts cited": sum(label.startswith("G") for label in cited),
            "words": len(result["answer"].split())}


if len(answers) == 2 * len(EVAL_SET):
    answer_checks = pd.DataFrame([checks(r) for r in answers.values()])
    answer_checks.to_csv(OUT_DIR / "answer_checks.csv", index=False)
    names = {"text": "text only (WP1)", "graph": "text + graph"}
    columns = ["cites a source", "cites the source article", "citation coverage", "unknown citations", "no answer", "words"]
    display(answer_checks.groupby(answer_checks["system"].map(names))[columns].mean().round(3))
    display(answer_checks.pivot_table(index="type", columns=answer_checks["system"].map(names),
                                      values="cites the source article", aggfunc="mean").round(2))
    with_facts = answer_checks[(answer_checks["system"] == "graph") & answer_checks["graph facts given"].gt(0)]
    print(f"Graph answers with graph facts in the prompt: {len(with_facts)}; citing at least one of them: "
          f"{with_facts['graph facts cited'].gt(0).sum()}")

,cites a source,cites the source article,citation coverage,unknown citations,no answer,words
system,,,,,,
text + graph,0.981,0.779,0.75,0.0,0.169,104.089
text only (WP1),0.981,0.775,0.72,0.0,0.211,105.315


system,text + graph,text only (WP1)
type,,
analytical,0.81,0.78
comparative,0.74,0.79
factual,0.81,0.81
relational,0.85,0.79
temporal,0.66,0.68


Graph answers with graph facts in the prompt: 43; citing at least one of them: 19


In [ ]:
def side_by_side(question_id):
    q = next(q for q in EVAL_SET if q["id"] == question_id)
    text_result, graph_result = answers[("text", question_id)]["result"], answers[("graph", question_id)]["result"]
    facts = "; ".join(f"[{f['label']}] {f['subject']} -> {f['relation']} -> {f['object']} ({f['date']})"
                      for f in graph_result["facts"]) or "none"
    display(Markdown(f"**[{q['type']}] {q['question']}**  \n*Reference:* {q['reference_answer']}  \n"
                     f"**Text only:** {text_result['answer']}  \n**Text + graph:** {graph_result['answer']}  \n"
                     f"*Graph facts:* {facts}"))


if len(answers) == 2 * len(EVAL_SET):
    for question_id in with_facts.sort_values("id")["id"].head(4):
        side_by_side(question_id)

**[temporal] When did Google launch DataGemma?**  
*Reference:* On 12 September 2024: an article published that day reports that Google debuted DataGemma that day.  
**Text only:** The provided sources do not specify when Google launched DataGemma. While multiple sources discuss the introduction, features, and capabilities of DataGemma, they all reference it as being launched in September 2024 [S1][S3][S5][S6][S4][S2], but none state the exact date of the launch. Therefore, based on the information available, the specific launch date of DataGemma cannot be determined.  
**Text + graph:** The provided sources do not specify when Google launched DataGemma. While several sources mention the launch or announcement of DataGemma and its features, they do not include a specific date for the launch. 

[S1], [S3], [S5], [S4], and [S6] all reference the launch or announcement of DataGemma in the context of 2024, with publication dates ranging from September 12 to September 18, 2024. However, these dates are publication dates of articles discussing DataGemma, not the actual launch date of the model.

Therefore, based on the available information, the exact launch date of DataGemma cannot be determined. 

[Note: The graph fact [G1] states that Gemma was developed by Google on 2024-09-18, but this does not confirm the launch date of DataGemma, which is a separate model.]  
*Graph facts:* [G1] Gemma -> developed by -> Google (2024-09-18)

**[relational] Which model family is DataGemma built on?**  
*Reference:* Google's Gemma family of open models; DataGemma grounds its answers in data from Google's Data Commons platform.  
**Text only:** DataGemma is built on the Gemma family of open models [S1].  
**Text + graph:** DataGemma is built on the Gemma family of models [S1][S5]. This is supported by the fact that DataGemma "builds on the existing Gemma family of open models" and leverages the same research and technology used to create the Gemini models, which is the foundation of the Gemma family [S5]. Additionally, the graph fact [G1][G2][G3][G4] confirms that multiple models in the Gemma family are part of the same foundational model lineage, and DataGemma is explicitly linked to this family through its design and training foundation. Therefore, the model family that DataGemma is built on is the Gemma family.  
*Graph facts:* [G1] Gemma-3 1B -> member of family -> Gemma (2026-03-31); [G2] Gemma-2-27B -> member of family -> Gemma (2024-09-18); [G3] Gemma 2B -> member of family -> Gemma (2025-02-19); [G4] MedGemma 27B -> member of family -> Gemma (2025-07-09)

**[factual] How does Google plan to improve DataGemma?**  
*Reference:* By training it on more information and increasing its question-answering capacity from a few hundred to millions of questions.  
**Text only:** Google plans to improve DataGemma by training the model on more information and increasing its question-answering capacity from a few hundred to millions of parameters [S1]. This enhancement aims to improve factual accuracy and reduce hallucinations, especially in statistical queries [S6]. The model leverages Google’s Data Commons, which contains over 240 billion data points, to ground its responses in real-world data [S6]. Additionally, DataGemma uses a large context window—up to 128,000 tokens with potential scaling to 1 million tokens—enabling it to hold and process more information, which helps minimize hallucinations by allowing cross-checking of responses against real data [S2]. Google also intends to refine the RAG and RIG methodologies through rigorous testing and eventually integrate these improvements into both Gemma and Gemini models via a phased, limited-access approach [S5].  
**Text + graph:** Google plans to improve DataGemma by training the model on more information and increasing its question-answering capacity from a few hundred to millions of parameters, which will enhance its ability to retrieve and utilize real-world data for more accurate and grounded responses [S1]. This improvement is part of a broader effort to reduce AI hallucinations by leveraging retrieval-augmented generation (RAG) and integrating extensive real-world data from the Data Commons platform [S3][S4][S6]. The use of a large context window—up to 128,000 tokens with potential scalability to 1 million tokens—allows DataGemma to hold and process more relevant information in memory, enabling it to cross-check its outputs with factual data and thus minimize hallucinations [S5]. Additionally, Google intends to refine the RAG and RIG methodologies through rigorous testing and will eventually integrate these enhancements into both Gemma and Gemini models via a phased, limited-access rollout [S2].  
*Graph facts:* [G1] Gemma -> developed by -> Google (2024-09-18)

**[relational] Who at Google presented DataGemma as an early step in grounded AI?**  
*Reference:* Google software engineer Jennifer Chen and Prem Ramaswami, the head of Data Commons, in a blog post.  
**Text only:** Jennifer Chen and Prem Ramaswami, the head of Data Commons, presented DataGemma as an early step in grounded AI [S1].  
**Text + graph:** Jennifer Chen and Prem Ramaswami presented DataGemma as an early step in grounded AI [S1].  
*Graph facts:* [G1] Gemma -> developed by -> Google (2024-09-18)

## 6. Saved Outputs

| File (`OUT_DIR`) | Content |
|---|---|
| `retrieval_results.csv` | per question: route, linked entities, rank of the source article with text only and with text + graph, graph facts in the prompt |
| `answers.jsonl` | the answers of both systems, with the full `synthesize_answer` result (sources, graph facts, citation checks) |
| `answer_checks.csv` | the citation checks of Section 5.2 per answer |
| `graph_rag_settings.json` | the settings of the graph-enhanced system, next to `rag_settings.json` of WP1 |

In [ ]:
retrieval_results = pd.DataFrame([{
    "id": q["id"], "type": q["type"], "question": q["question"], "route": runs[q["id"]]["route"],
    "entities": "; ".join(e["name"] for e in runs[q["id"]]["entities"]),
    "intents": "; ".join(runs[q["id"]]["intents"]),
    "rr_text": rank_text[q["id"]], "rr_text_graph": rank_graph[q["id"]],
} for q in EVAL_SET])
retrieval_results.to_csv(OUT_DIR / "retrieval_results.csv", index=False)

from stage3_graph_rag import fusion
GRAPH_RAG_SETTINGS = {
    "baseline": "rag_settings.json of WP1 (corpus, chunks, embeddings, retrieval, generation, model)",
    "knowledge_graph": {"nodes": "kg_v2/nodes.parquet", "relations": "kg_v2/typed_edges_assertions.parquet",
                        "relations_used": len(assertions), "entity_mentions": "stage1_wp2/entity_mentions.csv"},
    "linking": {"named_types": sorted(NAMED_TYPES), "family_patterns": "WP1 lexicon", "exceptions": FAMILY_EXCEPTIONS},
    "fusion": {"rankings": ["text", "entity", "relation"], "rrf_k": fusion.RRF_K, "entity_pool": fusion.POOL_SIZE,
               "size": RETRIEVAL_ARGS["size"]},
    "prompt": {"graph_facts": "evidence chunk among the prompt passages", "max_facts": graph_rag.max_facts,
               "lexicon_facts": "not used"},
    "intents": sorted(fusion.INTENTS),
}
(OUT_DIR / "graph_rag_settings.json").write_text(json.dumps(GRAPH_RAG_SETTINGS, indent=2), encoding="utf-8")
print(json.dumps(GRAPH_RAG_SETTINGS, indent=2))

{
  "baseline": "rag_settings.json of WP1 (corpus, chunks, embeddings, retrieval, generation, model)",
  "knowledge_graph": {
    "nodes": "kg_v2/nodes.parquet",
    "relations": "kg_v2/typed_edges_assertions.parquet",
    "relations_used": 4190,
    "entity_mentions": "stage1_wp2/entity_mentions.csv"
  },
  "linking": {
    "named_types": [
      "AI_MODEL",
      "BENCHMARK",
      "COMPANY",
      "DATASET",
      "FOUNDATION_MODEL",
      "GPE",
      "HARDWARE",
      "LAW",
      "LICENSE",
      "ORG",
      "PERSON",
      "PLATFORM",
      "PRODUCT",
      "STANDARD",
      "TECHNOLOGY"
    ],
    "family_patterns": "WP1 lexicon",
    "exceptions": {
      "Llama": "\\bLlama[\\w.\\- ]{0,15}?(?:Nemotron|Stack)\\b|\\bLlamaCon\\b"
    }
  },
  "fusion": {
    "rankings": [
      "text",
      "entity",
      "relation"
    ],
    "rrf_k": 60,
    "entity_pool": 100,
    "size": 30
  },
  "prompt": {
    "graph_facts": "evidence chunk among the prompt passages",
    "max_facts": 5

## AI Assistance Disclosure
- **Coding assistant**: Claude (Anthropic) was used to write and debug the code of the entity linking, the graph-aware
  retrieval and this notebook.
- **Writing**: Claude helped shorten and rephrase the markdown texts. All interpretations are based on the outputs of
  the notebook and were checked by the team member.